# ¿Qué determina la valoración de un perfume?
### Análisis de ~24.000 fragancias de Fragrantica: calidad percibida, demanda y posicionamiento de mercado

**Pregunta de negocio.** Si una marca quisiera lanzar un perfume, ¿qué características se asocian a una mejor valoración
de la comunidad, y esa valoración se traduce en demanda?

**Estructura**
1. Carga y auditoría de calidad de datos (con decisiones documentadas)
2. Construcción de la variable objetivo y de las variables explicativas
3. Exploración de datos
4. **Parte 1 — ¿Qué determina la valoración?** (modelos predictivos + regresión interpretable + sensibilidad)
5. **Parte 2 — Mercado: valoración vs. demanda** (concentración, segmentos, marcas, oferta por acorde)
6. Conclusiones y limitaciones

**Datos.** Dos archivos de Fragrantica: `fra_perfumes.csv` (catálogo completo, ~70.000 perfumes) y `fra_cleaned.csv`
(subconjunto enriquecido de ~24.000 perfumes con marca, país, año, notas y perfumistas).

**Cómo ejecutarlo.** Colocar ambos CSV en `./data/` y correr todo el notebook (≈6 min). Todos los parámetros ajustables
están en la celda de configuración de abajo.

In [ ]:
import re
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

## 0. Configuración
Todos los valores marcados acá se pueden cambiar y re-ejecutar el notebook completo.

In [ ]:
# ----------------------------- CONFIGURACIÓN -----------------------------
DATA_DIR = Path("data")
FILE_CLEAN = DATA_DIR / "fra_cleaned.csv"      # subconjunto enriquecido (24k perfumes)
FILE_FULL = DATA_DIR / "fra_perfumes.csv"      # catálogo completo (70k perfumes)
FIG_DIR = Path("figures")

RANDOM_STATE = 42
SHRINK_M = 50          # peso del "prior" bayesiano, expresado en votos (ver sección 2)
RATING_FLOOR = 3.0     # perfumes con rating crudo < este valor se excluyen del modelado (anomalías)
MIN_PREV = 0.01        # prevalencia mínima para conservar una nota / acorde como variable
MIN_COUNTRY = 100      # países con menos perfumes se agrupan en "Other"
N_SPLITS = 5           # folds de validación cruzada
PERM_REPEATS = 5       # repeticiones de la importancia por permutación
Q_QUALITY = 0.75       # percentil de valoración que define "alta calidad"
Q_DEMAND = 0.75        # percentil de votos que define "alta demanda"
MIN_BRAND_PERFUMES = 20  # mínimo de perfumes para rankear una marca

HGB_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                  min_samples_leaf=40, l2_regularization=1.0, random_state=RANDOM_STATE)
# --------------------------------------------------------------------------

FIG_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False})
PAL = {"women": "#c2410c", "men": "#1d4ed8", "unisex": "#15803d"}


def save(fig, name):
    """Guarda la figura en ./figures y la muestra."""
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()


def titulo(s):
    """'jean-paul-gaultier' -> 'Jean Paul Gaultier'"""
    return s.str.replace("-", " ", regex=False).str.title()

## 1. Carga y auditoría de calidad de datos

In [ ]:
# fra_cleaned.csv: separador ';', decimales con coma y codificación cp1252 (no UTF-8).
raw = pd.read_csv(FILE_CLEAN, sep=";", encoding="cp1252")
full = pd.read_csv(FILE_FULL)

print(f"fra_cleaned.csv : {raw.shape[0]:,} filas x {raw.shape[1]} columnas")
print(f"fra_perfumes.csv: {full.shape[0]:,} filas x {full.shape[1]} columnas")
print("\nColumnas de fra_cleaned :", list(raw.columns))
print("Columnas de fra_perfumes:", list(full.columns))

In [ ]:
# --- Normalización de fra_cleaned ---
cl = raw.rename(columns={"Rating Value": "rating", "Rating Count": "votes"})
cl.columns = [c.lower() for c in cl.columns]
cl["rating"] = cl["rating"].str.replace(",", ".", regex=False).astype(float)
cl["pid"] = cl["url"].str.extract(r"-(\d+)\.html$")[0].astype("int64")  # id de Fragrantica

# --- Normalización de fra_perfumes ---
full["pid"] = pd.to_numeric(full["url"].str.extract(r"-(\d+)\.html$")[0]).astype("Int64")
full["votes"] = pd.to_numeric(full["Rating Count"].str.replace(",", "", regex=False), errors="coerce")
full["gender"] = full["Gender"].map({"for women": "women", "for men": "men", "for women and men": "unisex"})
full = full.rename(columns={"Rating Value": "rating"})
assert full["Rating Count"].isna().sum() == full["votes"].isna().sum(), "hay votos no parseables"

# fra_perfumes trae filas repetidas (mismo ID de perfume): se conserva la primera.
n_antes = len(full)
full = full.dropna(subset=["pid"]).drop_duplicates("pid").reset_index(drop=True)
print(f"fra_perfumes: {n_antes - len(full)} filas repetidas eliminadas -> {len(full):,} perfumes únicos")

### 1.1 Valores faltantes y columnas problemáticas

In [ ]:
aud = pd.DataFrame({
    "nulos": cl.isna().sum(),
    "% nulos": (cl.isna().mean() * 100).round(1),
})
aud.loc["perfumer1 == 'unknown'", ["nulos", "% nulos"]] = [
    (cl["perfumer1"] == "unknown").sum(), round((cl["perfumer1"] == "unknown").mean() * 100, 1)]
aud = aud[aud["nulos"] > 0]
display(aud)

print(f"\nIDs de perfume duplicados en fra_cleaned: {cl['pid'].duplicated().sum()}")
dup_nombre = cl.duplicated(["brand", "perfume"], keep=False).sum()
print(f"Filas con mismo marca+nombre pero distinto ID: {dup_nombre} "
      f"(son entradas distintas en Fragrantica —versiones/concentraciones— y se conservan)")
print(f"Perfumes anteriores a 1950: {(cl['year'] < 1950).sum()} (clásicos reales; se conservan)")

### 1.2 Relación entre los dos archivos
Antes de decidir cómo usarlos, verifico si son independientes o si uno contiene al otro.

In [ ]:
ids_full = set(full["pid"].dropna().astype("int64"))
es_subconjunto = cl["pid"].isin(ids_full).all()
print(f"¿Todos los IDs de fra_cleaned están en fra_perfumes? -> {es_subconjunto}")

m = cl.merge(full.dropna(subset=["pid"]).drop_duplicates("pid").assign(pid=lambda d: d["pid"].astype("int64")),
             on="pid", suffixes=("_c", "_f"))
print(f"IDs cruzados: {len(m):,}")
print(f"Máx. diferencia absoluta de rating : {(m['rating_c'] - m['rating_f']).abs().max():.3f}")
print(f"Máx. diferencia absoluta de votos  : {(m['votes_c'] - m['votes_f']).abs().max():.0f}")

**Conclusión:** `fra_cleaned` es un **subconjunto** del catálogo completo (ratings y votos prácticamente idénticos), no una fuente independiente.
Por lo tanto no sirve para "validar" un archivo con el otro. Uso cada uno para lo que aporta:
- `fra_cleaned` → modelado (tiene marca, país, año, notas y perfumistas).
- `fra_perfumes` → contexto de mercado sobre todo el catálogo (concentración de la demanda, sección 5.1).

Falta verificar que el subconjunto no esté sesgado respecto del catálogo en su mismo rango de votos:

In [ ]:
min_votes = cl["votes"].min()
cat = full[full["votes"] >= min_votes].copy()
cat["en_cleaned"] = cat["pid"].isin(cl["pid"])
comp = cat.groupby("en_cleaned").agg(
    perfumes=("rating", "size"),
    rating_medio=("rating", "mean"),
    votos_mediana=("votes", "median"),
    pct_women=("gender", lambda s: (s == "women").mean() * 100),
    pct_men=("gender", lambda s: (s == "men").mean() * 100),
    pct_unisex=("gender", lambda s: (s == "unisex").mean() * 100),
).round(2)
comp.index = ["Fuera de fra_cleaned", "Dentro de fra_cleaned"]
print(f"Perfumes del catálogo con ≥{min_votes} votos: {len(cat):,}; {cat['en_cleaned'].mean():.0%} están en fra_cleaned")
display(comp)

El subconjunto retiene ~3/4 de los perfumes con votos suficientes y el **rating medio es casi idéntico**, pero la composición
**no lo es**: dentro hay más perfumes femeninos y masculinos y bastante menos unisex, y con más votos (mediana más alta).
**`fra_cleaned` no es una muestra aleatoria del catálogo.** Las conclusiones aplican a perfumes con **≥26 votos** y a este
subconjunto; para no depender de ese supuesto, la sección 4.5 **replica** el análisis con los perfumes que quedaron afuera.

### 1.3 Anomalías en el rating

In [ ]:
bajos = cl[cl["rating"] < RATING_FLOOR].copy()
bajos["pride"] = bajos["perfume"].str.contains("pride", case=False)
print(f"Perfumes con rating < {RATING_FLOOR}: {len(bajos)} ({len(bajos) / len(cl):.2%} del total)")
print(f"  - con 'pride' en el nombre: {bajos['pride'].sum()}")
print(f"  - resto: {(~bajos['pride']).sum()}")
bajos_show = bajos.sort_values("rating").head(12).assign(
    marca=lambda d: titulo(d["brand"]), perfume=lambda d: titulo(d["perfume"]))[["marca", "perfume", "rating", "votes"]]
display(bajos_show.reset_index(drop=True))

var_total = ((cl["rating"] - cl["rating"].mean()) ** 2).sum()
var_bajos = ((bajos["rating"] - cl["rating"].mean()) ** 2).sum()
print(f"\nEstos {len(bajos)} perfumes aportan el {var_bajos / var_total:.1%} de la varianza total del rating.")

Un grupo de ediciones "Pride" concentra los ratings más bajos del dataset (1,4–2,5), muy por debajo del resto (mediana ≈ 4,0).
Ese patrón sugiere que parte de esos votos responden a motivos ajenos al aroma, por lo que no representan la misma señal que
el resto del catálogo. Como 64 perfumes (0,3 %) explican ~6 % de la varianza total (cifra impresa arriba), distorsionarían los modelos.

**Decisión:** se excluyen del **modelado** (`RATING_FLOOR = 3.0`, ajustable), pero se conservan en el análisis descriptivo
y se mide el efecto de esta decisión en la sección 4.4 (sensibilidad).

## 2. Variable objetivo y variables explicativas
### 2.1 El rating crudo es ruidoso cuando hay pocos votos
Un perfume con 26 votos y rating 4,8 no es comparable con uno con 20.000 votos y rating 4,3. El gráfico lo muestra:
la dispersión del rating se reduce a medida que aumentan los votos ("embudo").

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(cl["votes"], cl["rating"], s=4, alpha=0.25, color="#475569")
ax.axhline(cl["rating"].mean(), color="#c2410c", lw=1.5, ls="--", label=f"Promedio global ({cl['rating'].mean():.2f})")
ax.set_xscale("log")
ax.set_xlabel("Cantidad de votos (escala log)")
ax.set_ylabel("Rating crudo")
ax.set_title("Menos votos → rating más disperso (efecto embudo)")
ax.legend()
save(fig, "01_embudo_rating_votos")

**Solución:** uso un rating **suavizado bayesiano** (*shrinkage*), que acerca al promedio global los ratings con pocos votos:

`score = (votos × rating + m × promedio_global) / (votos + m)`

con `m = SHRINK_M = 50` (equivale a asumir un "prior" de 50 votos en el promedio). `m` es un hiperparámetro:
en 4.4 verifico que las conclusiones no dependen de ese valor.

In [ ]:
def shrunk(rating, votes, m, prior):
    """Rating bayesiano suavizado."""
    return (votes * rating + m * prior) / (votes + m)


PRIOR = cl["rating"].mean()
cl["score"] = shrunk(cl["rating"], cl["votes"], SHRINK_M, PRIOR)
cl["log_votes"] = np.log10(cl["votes"])

print(f"Promedio global (prior): {PRIOR:.3f}")
print(f"Desvío del rating crudo : {cl['rating'].std():.3f}")
print(f"Desvío del score suavizado: {cl['score'].std():.3f}")

### 2.2 Variables explicativas
- **Género** (women / men / unisex), **país de la marca**, **año** (con indicador de faltante).
- **Perfumista**: si está identificado o figura como `unknown`, y cuántos hay.
- **Acordes principales** y **notas** (salida + corazón + fondo): una columna 0/1 por cada acorde/nota que aparece en al menos
  el 1 % de los perfumes (`MIN_PREV`).
- Los **votos** *no* se usan como variable explicativa del rating: son un resultado conjunto (ver Parte 2) y mezclarlos
  con el target suavizado generaría una dependencia mecánica.
- La **marca** se trata aparte (4.1): es una variable de alta cardinalidad y la más fácil de usar para "hacer trampa".

In [ ]:
def split_notes(s):
    return {x.strip() for x in str(s).split(",") if x.strip()}


def multihot(sets, prefix, min_prev, index):
    cnt = Counter(x for s in sets for x in s)
    keep = sorted(k for k, v in cnt.items() if v / len(sets) >= min_prev)
    return pd.DataFrame({f"{prefix}_{k}": [int(k in s) for s in sets] for k in keep}, index=index)


def build_features(d):
    """Devuelve (X, grupos): matriz de variables y diccionario grupo -> columnas (para importancia por grupos)."""
    idx = d.index
    notes = [split_notes(a) | split_notes(b) | split_notes(c) for a, b, c in zip(d["top"], d["middle"], d["base"])]
    acc_cols = [f"mainaccord{i}" for i in range(1, 6)]
    accs = [set(r.dropna()) for _, r in d[acc_cols].iterrows()]

    G = pd.get_dummies(d["gender"], prefix="g").astype(int)
    big = d["country"].value_counts()
    big = big[big >= MIN_COUNTRY].index
    C = pd.get_dummies(d["country"].where(d["country"].isin(big), "Other"), prefix="ctry").astype(int)
    Y = pd.DataFrame({"year": d["year"].fillna(d["year"].median()), "year_na": d["year"].isna().astype(int)}, index=idx)
    P = pd.DataFrame({"perf_known": (d["perfumer1"] != "unknown").astype(int),
                      "n_perf": (d["perfumer1"] != "unknown").astype(int) + d["perfumer2"].notna().astype(int)}, index=idx)
    A = multihot(accs, "acc", MIN_PREV, idx)
    N = multihot(notes, "note", MIN_PREV, idx)
    N["n_notes"] = [len(s) for s in notes]

    X = pd.concat([G, C, Y, P, A, N], axis=1)
    grupos = {"Género": list(G.columns), "País": list(C.columns), "Época (año)": list(Y.columns),
              "Perfumista": list(P.columns), "Acordes": list(A.columns), "Notas": list(N.columns)}
    return X, grupos


# Conjunto de modelado: sin las anomalías de rating
df = cl[cl["rating"] >= RATING_FLOOR].reset_index(drop=True)
X, GRUPOS = build_features(df)
y_score = df["score"].to_numpy()
y_logv = df["log_votes"].to_numpy()
brand = df["brand"].to_numpy()

print(f"Perfumes en el modelado: {len(df):,} (excluidos: {len(cl) - len(df)})")
print(f"Variables: {X.shape[1]}  ->  " + ", ".join(f"{g}: {len(c)}" for g, c in GRUPOS.items()))
print(f"Marcas distintas: {df['brand'].nunique():,}")

## 3. Exploración de datos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(cl["rating"], bins=50, color="#475569", ax=axes[0])
axes[0].axvline(cl["rating"].median(), color="#c2410c", ls="--", label=f"Mediana {cl['rating'].median():.2f}")
axes[0].set(title="Distribución del rating", xlabel="Rating crudo", ylabel="Perfumes")
axes[0].legend()

sns.histplot(cl["votes"], bins=60, log_scale=(True, False), color="#475569", ax=axes[1])
axes[1].axvline(cl["votes"].median(), color="#c2410c", ls="--", label=f"Mediana {cl['votes'].median():.0f}")
axes[1].set(title="Distribución de votos (demanda)", xlabel="Votos (escala log)", ylabel="Perfumes")
axes[1].legend()

orden = ["women", "unisex", "men"]
sns.boxplot(data=df, x="gender", y="score", order=orden, palette=PAL, ax=axes[2], fliersize=1)
axes[2].set(title="Valoración por género", xlabel="", ylabel="Score suavizado")
fig.tight_layout()
save(fig, "02_distribuciones")

In [ ]:
# Efecto de la época (con intervalo de confianza 95 % de la media) y del país.
df["decada"] = (df["year"] // 10 * 10)
dec = df.dropna(subset=["decada"]).groupby("decada")["score"].agg(["mean", "sem", "count"])
dec = dec[dec["count"] >= 20]

pais = df.groupby("country")["score"].agg(["mean", "sem", "count"])
pais = pais[pais["count"] >= MIN_COUNTRY].sort_values("mean")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
axes[0].errorbar(dec.index, dec["mean"], yerr=1.96 * dec["sem"], fmt="o-", color="#1d4ed8", capsize=3)
axes[0].axhline(df["score"].mean(), color="#94a3b8", ls="--", lw=1)
axes[0].set(title="Valoración media por década de lanzamiento", xlabel="Década", ylabel="Score suavizado")
for x_, r in dec.iterrows():
    axes[0].annotate(f"n={int(r['count'])}", (x_, r["mean"]), textcoords="offset points", xytext=(0, 9),
                     ha="center", fontsize=7, color="#475569")

axes[1].barh(pais.index, pais["mean"], xerr=1.96 * pais["sem"], color="#15803d", alpha=0.85, capsize=2)
axes[1].axvline(df["score"].mean(), color="#94a3b8", ls="--", lw=1)
axes[1].set_xlim(3.6, 4.2)
axes[1].set(title=f"Valoración media por país (≥{MIN_COUNTRY} perfumes)", xlabel="Score suavizado")
fig.tight_layout()
save(fig, "03_epoca_pais")

## 4. Parte 1 — ¿Qué determina la valoración?

### 4.1 Protocolo de validación (la parte más importante para que el resultado sea honesto)
Muchos perfumes comparten marca, y la marca por sí sola es muy informativa. Si se reparte al azar entre entrenamiento
y prueba, el modelo "reconoce" la marca y el resultado se infla. Por eso comparo tres escenarios:

| Escenario | Qué responde |
|---|---|
| **A. Marcas nuevas** (`GroupKFold` por marca, sin variable de marca) | ¿Qué tanto se puede anticipar la valoración **solo por las características del perfume**? |
| **B. Reparto aleatorio sin marca** | Referencia para mostrar cuánto se infla la métrica al mezclar marcas |
| **C. Marca conocida** (reparto aleatorio + codificación de marca fuera de muestra) | ¿Cuánto mejora si además se **conoce la marca**? |

La codificación de marca (*target encoding*) se calcula **solo con datos de entrenamiento** y, dentro de ellos, de forma anidada
(por *folds* internos), para que ningún perfume use su propio rating en su variable de marca.

Métrica principal: **R² fuera de muestra** (proporción de la varianza del score que el modelo logra predecir; 0 = no mejora al promedio).

In [ ]:
def make_ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 20)))


def make_hgb():
    return HistGradientBoostingRegressor(**HGB_PARAMS)


def make_dummy():
    return DummyRegressor(strategy="mean")


def brand_te_oof(b_tr, y_tr, k=10, inner=5):
    """Target encoding de marca para el set de entrenamiento, calculado fuera de muestra (folds internos).
    Devuelve (te_train, n_train, mapa_te_completo, mapa_conteo, promedio_global)."""
    gm = y_tr.mean()
    te, nn = np.zeros(len(y_tr)), np.zeros(len(y_tr))
    for a, b in KFold(inner, shuffle=True, random_state=RANDOM_STATE).split(b_tr):
        st = pd.DataFrame({"b": b_tr[a], "y": y_tr[a]}).groupby("b")["y"].agg(["sum", "count"])
        s = pd.Series(b_tr[b]).map(st["sum"]).fillna(0).to_numpy()
        n = pd.Series(b_tr[b]).map(st["count"]).fillna(0).to_numpy()
        te[b], nn[b] = (s + k * gm) / (n + k), n
    st = pd.DataFrame({"b": b_tr, "y": y_tr}).groupby("b")["y"].agg(["sum", "count"])
    return te, nn, (st["sum"] + k * gm) / (st["count"] + k), st["count"], gm


def cv_run(make_model, Xm, y, splitter, groups=None, use_brand=False):
    """Validación cruzada. Devuelve dict con R² y MAE fuera de muestra."""
    pred = np.zeros(len(y))
    r2_folds = []
    for tr, te in splitter.split(Xm, y, groups):
        Xtr, Xte = Xm.iloc[tr].copy(), Xm.iloc[te].copy()
        if use_brand:
            t_tr, n_tr, full_map, cnt_map, gm = brand_te_oof(brand[tr], y[tr])
            Xtr["brand_te"], Xtr["brand_n"] = t_tr, n_tr
            Xte["brand_te"] = pd.Series(brand[te]).map(full_map).fillna(gm).to_numpy()
            Xte["brand_n"] = pd.Series(brand[te]).map(cnt_map).fillna(0).to_numpy()
        model = make_model().fit(Xtr, y[tr])
        pred[te] = model.predict(Xte)
        r2_folds.append(r2_score(y[te], pred[te]))
    return {"R² (fuera de muestra)": r2_score(y, pred), "MAE": mean_absolute_error(y, pred),
            "R² medio por fold": np.mean(r2_folds), "Desvío entre folds": np.std(r2_folds)}


gkf = GroupKFold(n_splits=N_SPLITS)
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

resultados = {
    "Baseline: promedio": cv_run(make_dummy, X, y_score, gkf, brand),
    "A1. Ridge · marcas nuevas": cv_run(make_ridge, X, y_score, gkf, brand),
    "A2. Gradient Boosting · marcas nuevas": cv_run(make_hgb, X, y_score, gkf, brand),
    "B. Gradient Boosting · reparto aleatorio, sin marca": cv_run(make_hgb, X, y_score, kf),
    "C. Gradient Boosting · marca conocida": cv_run(make_hgb, X, y_score, kf, use_brand=True),
}
tabla_modelos = pd.DataFrame(resultados).T.round(3)
display(tabla_modelos)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
nombres = ["Baseline", "A1 Ridge\n(marcas nuevas)", "A2 GBM\n(marcas nuevas)", "B GBM\n(azar, sin marca)", "C GBM\n(marca conocida)"]
vals = tabla_modelos["R² (fuera de muestra)"].to_numpy()
cols = ["#94a3b8", "#1d4ed8", "#1d4ed8", "#f59e0b", "#15803d"]
bars = ax.bar(nombres, vals, color=cols)
for b_, v_ in zip(bars, vals):
    ax.text(b_.get_x() + b_.get_width() / 2, v_ + 0.006, f"{v_:.2f}", ha="center", fontsize=10, fontweight="bold")
ax.set(ylabel="R² fuera de muestra", title="¿Cuánto de la valoración se puede predecir?", ylim=(0, max(vals) * 1.2))
fig.tight_layout()
save(fig, "04_r2_modelos")

**Lectura**
- Con **marcas nuevas** (A), las características del perfume explican una fracción modesta de la valoración (≈12–15 %).
  El modelo no lineal queda apenas por encima del lineal (A2 vs. A1), pero la diferencia es chica frente a la variación entre
  folds (≈0,03): no la tomo como concluyente.
- Repartir al azar **sin** usar la marca (B) ya da una métrica más alta: es la "fuga" por parecidos entre perfumes de la misma marca.
  Por eso el escenario A es el que usaremos para decir qué determina la valoración.
- Conocer la **marca** (C) duplica la capacidad predictiva respecto de A2: agregar la marca suma tanto poder predictivo como todas las demás variables juntas.
- Aun así, la mayor parte de la varianza queda sin explicar: el gusto individual y el ruido de votación son grandes.

### 4.2 ¿Qué grupo de variables importa más? (importancia por permutación, marcas nuevas)
Para cada fold se entrena el modelo, y en el conjunto de prueba se **mezclan al azar todas las columnas de un grupo a la vez**
(por ejemplo, todos los acordes). Cuánto cae el R² mide cuánto aportaba ese grupo.
Como los grupos están correlacionados, las caídas **no suman** el R² total: sirven para ordenar, no para repartir porcentajes.

In [ ]:
def group_importance(Xm, y, grupos, reps=PERM_REPEATS):
    rng = np.random.RandomState(RANDOM_STATE)
    caidas = {g: [] for g in grupos}
    for tr, te in gkf.split(Xm, y, brand):
        model = make_hgb().fit(Xm.iloc[tr], y[tr])
        Xte = Xm.iloc[te]
        base = r2_score(y[te], model.predict(Xte))
        for g, cols_ in grupos.items():
            d = []
            for _ in range(reps):
                Xp = Xte.copy()
                Xp[cols_] = Xp[cols_].to_numpy()[rng.permutation(len(Xp))]
                d.append(base - r2_score(y[te], model.predict(Xp)))
            caidas[g].append(np.mean(d))
    return pd.DataFrame({"caida_R2": {g: np.mean(v) for g, v in caidas.items()},
                         "sd_folds": {g: np.std(v) for g, v in caidas.items()}})


imp_score = group_importance(X, y_score, GRUPOS).sort_values("caida_R2", ascending=False)
display(imp_score.round(4))

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(imp_score.index[::-1], imp_score["caida_R2"][::-1], xerr=imp_score["sd_folds"][::-1], color="#1d4ed8", capsize=3)
ax.set(xlabel="Caída del R² al mezclar el grupo (mayor = más importante)", title="Importancia por grupo de variables — valoración")
fig.tight_layout()
save(fig, "05_importancia_valoracion")

Acordes, país y género encabezan el ranking, pero las diferencias entre ellos están dentro del ruido entre folds (barras de error);
época y notas vienen después, y el perfumista casi no aporta a la valoración.

### 4.3 ¿En qué dirección empuja cada variable? Regresión lineal con errores agrupados por marca
El modelo predictivo ordena la importancia, pero no dice si un acorde sube o baja la valoración. Para eso uso una regresión
lineal (MCO) con **errores estándar agrupados por marca** (perfumes de una misma marca no son independientes; ignorarlo
haría parecer "significativo" casi todo).

Se ajusta dos veces: (1) **sin** efectos de marca; (2) **con efectos fijos de marca**, que compara perfumes *dentro* de la
misma marca. Si un efecto sobrevive en (2), no es un artefacto de que cierta marca use cierto acorde.
Categorías de referencia: género = women, década = 2010s, país = France.

In [ ]:
d_ols = df.copy()
safe = {c: re.sub(r"\W", "_", c) for c in X.columns}
Xs = X.rename(columns=safe)
acc_safe = [safe[c] for c in GRUPOS["Acordes"]]
d_ols = pd.concat([d_ols, Xs[acc_safe + ["perf_known"]]], axis=1)
d_ols["decada_g"] = pd.cut(d_ols["year"], [0, 1979, 1989, 1999, 2009, 2019, 2030],
                           labels=["pre1980", "1980s", "1990s", "2000s", "2010s", "2020s"]).astype(object)
d_ols["decada_g"] = d_ols["decada_g"].fillna("sin_anio")
big_c = d_ols["country"].value_counts()
big_c = big_c[big_c >= MIN_COUNTRY].index
d_ols["pais_g"] = d_ols["country"].where(d_ols["country"].isin(big_c), "Other")

formula = ('score ~ C(gender, Treatment("women")) + C(decada_g, Treatment("2010s")) '
           '+ C(pais_g, Treatment("France")) + perf_known + ' + " + ".join(acc_safe))
cluster = {"groups": pd.factorize(d_ols["brand"])[0]}
ols_sin = smf.ols(formula, d_ols).fit(cov_type="cluster", cov_kwds=cluster)
ols_fe = smf.ols(formula + " + C(brand)", d_ols).fit(cov_type="cluster", cov_kwds=cluster)
print(f"R² ajustado — sin efectos de marca: {ols_sin.rsquared_adj:.3f} | con efectos fijos de marca: {ols_fe.rsquared_adj:.3f}")


def coef_df(model, nombres):
    p, ci = model.params, model.conf_int()
    filas = []
    for etiqueta, clave in nombres.items():
        if clave in p.index:
            filas.append((etiqueta, p[clave], ci.loc[clave, 0], ci.loc[clave, 1]))
    return pd.DataFrame(filas, columns=["variable", "coef", "lo", "hi"]).set_index("variable")


g = 'C(gender, Treatment("women"))[T.{}]'
dc = 'C(decada_g, Treatment("2010s"))[T.{}]'
nom_factores = {"Género: men (vs women)": g.format("men"), "Género: unisex (vs women)": g.format("unisex"),
                "Década: pre-1980 (vs 2010s)": dc.format("pre1980"), "Década: 1980s": dc.format("1980s"),
                "Década: 1990s": dc.format("1990s"), "Década: 2000s": dc.format("2000s"),
                "Década: 2020s": dc.format("2020s"), "Perfumista identificado": "perf_known"}
f_sin = coef_df(ols_sin, nom_factores)
f_fe = coef_df(ols_fe, nom_factores)
tabla_factores = f_sin.join(f_fe, lsuffix=" (sin FE)", rsuffix=" (con FE marca)").round(3)
display(tabla_factores)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
ypos = np.arange(len(f_sin))[::-1]
ax.errorbar(f_sin["coef"], ypos + 0.15, xerr=[f_sin["coef"] - f_sin["lo"], f_sin["hi"] - f_sin["coef"]],
            fmt="o", color="#1d4ed8", capsize=3, label="Sin efectos de marca")
ax.errorbar(f_fe["coef"], ypos - 0.15, xerr=[f_fe["coef"] - f_fe["lo"], f_fe["hi"] - f_fe["coef"]],
            fmt="s", color="#15803d", capsize=3, label="Dentro de la misma marca")
ax.axvline(0, color="#64748b", lw=1)
ax.set_yticks(ypos)
ax.set_yticklabels(f_sin.index)
ax.set(xlabel="Diferencia de score (IC 95 %, errores agrupados por marca)", title="Género, época y perfumista")
ax.legend(loc="lower right")
fig.tight_layout()
save(fig, "06_efectos_genero_epoca")

In [ ]:
# Efecto de cada acorde principal (controlando género, época, país y perfumista).
inv = {v: k.replace("acc_", "") for k, v in safe.items() if k.startswith("acc_")}
a_sin = coef_df(ols_sin, {inv[c]: c for c in acc_safe})
a_fe = coef_df(ols_fe, {inv[c]: c for c in acc_safe})
corr_acc = a_sin["coef"].corr(a_fe["coef"])
print(f"Correlación entre los efectos de los {len(a_sin)} acordes con y sin efectos de marca: {corr_acc:.2f}")

a_plot = a_sin.sort_values("coef")
sel = pd.concat([a_plot.head(8), a_plot.tail(8)])
fig, ax = plt.subplots(figsize=(8, 5.6))
colores = ["#c2410c" if v < 0 else "#15803d" for v in sel["coef"]]
ax.barh(sel.index, sel["coef"], xerr=[sel["coef"] - sel["lo"], sel["hi"] - sel["coef"]], color=colores, capsize=2, alpha=0.9)
ax.axvline(0, color="#64748b", lw=1)
ax.set(xlabel="Diferencia de score vs. perfumes sin ese acorde (IC 95 %)",
       title="Acordes asociados a mayor y menor valoración (16 extremos de 44)")
fig.tight_layout()
save(fig, "07_efectos_acordes")

**Lectura**
- **Género:** en este subconjunto, a igual época, país y acordes, los perfumes masculinos y unisex puntúan algo más que los femeninos,
  y el efecto se mantiene dentro de la misma marca. **Pero la réplica (4.5) lo confirma solo a medias**: "masculino > femenino" se
  repite con menor magnitud; "unisex > femenino" no.
- **Época:** los perfumes anteriores a 1990 puntúan más que los recientes. Ojo: es probable un **sesgo de supervivencia**
  (solo los clásicos buenos siguen en el catálogo con votos suficientes), no que "antes se hacía mejor".
- **Perfumista identificado:** sin efecto sobre la valoración (pero sí sobre la demanda, ver Parte 2).
- **Acordes:** los cálidos/envolventes (tabaco, vainilla, oud, ámbar, canela) puntúan más; los acuáticos/marinos, afrutados,
  almizclados y lavanda, menos. Los efectos se sostienen dentro de la marca (correlación alta entre ambos ajustes).
  Con 44 acordes y IC al 95 %, cabe esperar un par de falsos positivos; la réplica (4.5) indica cuáles son más firmes.
- Son diferencias **pequeñas** (de 0,03 a 0,15 puntos sobre 5): relevantes en promedio, no determinantes para un perfume individual.
  Reflejan además la preferencia de la **comunidad de Fragrantica**, no necesariamente la del público general.

In [ ]:
# Efecto de país (los países son atributo de la marca: no se pueden estimar "dentro de la marca").
nom_pais = {p: f'C(pais_g, Treatment("France"))[T.{p}]' for p in sorted(big_c) if p != "France"}
nom_pais["Other"] = 'C(pais_g, Treatment("France"))[T.Other]'
pais_ef = coef_df(ols_sin, nom_pais).sort_values("coef")
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.errorbar(pais_ef["coef"], pais_ef.index, xerr=[pais_ef["coef"] - pais_ef["lo"], pais_ef["hi"] - pais_ef["coef"]],
            fmt="o", color="#475569", capsize=3)
ax.axvline(0, color="#64748b", lw=1)
ax.set(xlabel="Diferencia de score vs. Francia (IC 95 %)", title="Efecto del país de la marca (controlando el resto)")
fig.tight_layout()
save(fig, "08_efecto_pais")

### 4.4 Sensibilidad: ¿las conclusiones dependen de mis decisiones de limpieza?
Repito el escenario A (marcas nuevas, Gradient Boosting) cambiando una decisión por vez.

In [ ]:
def run_variant(d, target):
    """Escenario A (marcas nuevas, Gradient Boosting) sobre una variante del dataset."""
    d = d.reset_index(drop=True)
    Xv, _ = build_features(d)
    r = cv_run(make_hgb, Xv, d[target].to_numpy(), gkf, d["brand"].to_numpy())
    return r["R² (fuera de muestra)"], r["MAE"]


sens = {}
base_d = cl[cl["rating"] >= RATING_FLOOR].reset_index(drop=True)
sens["Base (piso 3,0 · m=50 · score)"] = run_variant(base_d, "score")
sens["Sin excluir anomalías (rating<3 incluido)"] = run_variant(cl.reset_index(drop=True), "score")
sens["Target = rating crudo (sin suavizar)"] = run_variant(base_d, "rating")
for m_ in (25, 100):
    tmp = base_d.copy()
    tmp["score_m"] = shrunk(tmp["rating"], tmp["votes"], m_, PRIOR)
    sens[f"Suavizado con m={m_}"] = run_variant(tmp, "score_m")
tabla_sens = pd.DataFrame(sens, index=["R² (marcas nuevas)", "MAE"]).T.round(3)
display(tabla_sens)

El R² (marcas nuevas) se mantiene entre ≈0,13 y ≈0,15 en todas las variantes: la conclusión de que los atributos del perfume
explican una parte modesta de la valoración **no depende** de estas decisiones. Excluir las anomalías y suavizar el target
mejoran levemente el ajuste, como era de esperar. El MAE **no es comparable** entre variantes con distinto target (cada uno
tiene otra escala y varianza). La dirección de los efectos (4.3) tampoco depende de ellas.

### 4.5 Réplica fuera del subconjunto (validez externa)
`fra_cleaned` no es una muestra aleatoria (sección 1.2). Para comprobar si los hallazgos dependen de esa selección, repito
la regresión en los perfumes del catálogo con ≥26 votos que **no están** en `fra_cleaned`, usando solo lo que el archivo grande
permite: género, década (extraída de la descripción), los **5 primeros acordes** (coinciden con los de `fra_cleaned` en el 99,8 %
de los perfumes en común) y la marca (tomada de la URL, para los errores agrupados).
Para comparar en igualdad de condiciones, se reajusta el modelo del subconjunto con **las mismas variables** (sin país ni perfumista,
que no existen afuera).

In [ ]:
import ast

fuera = full[(full["votes"] >= min_votes) & ~full["pid"].isin(cl["pid"]) & full["rating"].notna() & full["gender"].notna()].copy()
fuera = fuera[fuera["rating"] >= RATING_FLOOR].reset_index(drop=True)
fuera["brand"] = fuera["url"].str.extract(r"/perfume/([^/]+)/")[0].str.lower()
fuera["score"] = shrunk(fuera["rating"], fuera["votes"], SHRINK_M, PRIOR)
fuera["year"] = fuera["Description"].str.extract(r"launched in (\d{4})")[0].astype(float)
fuera["decada_g"] = pd.cut(fuera["year"], [0, 1979, 1989, 1999, 2009, 2019, 2030],
                           labels=["pre1980", "1980s", "1990s", "2000s", "2010s", "2020s"]).astype(object).fillna("sin_anio")
acc_afuera = fuera["Main Accords"].map(lambda s_: set(ast.literal_eval(s_)[:5]))
for col in GRUPOS["Acordes"]:
    nombre = col[len("acc_"):]
    fuera[safe[col]] = [int(nombre in s_) for s_ in acc_afuera]

form_rep = ('score ~ C(gender, Treatment("women")) + C(decada_g, Treatment("2010s")) + ' + " + ".join(acc_safe))


def fit_cluster(d):
    return smf.ols(form_rep, d).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d["brand"])[0]})


rep_in, rep_out = fit_cluster(d_ols), fit_cluster(fuera)
print(f"Perfumes — dentro de fra_cleaned: {len(d_ols):,} ({d_ols['brand'].nunique()} marcas) | "
      f"fuera: {len(fuera):,} ({fuera['brand'].nunique()} marcas)")
print(f"R² ajustado — dentro: {rep_in.rsquared_adj:.3f} | fuera: {rep_out.rsquared_adj:.3f}")

nom_rep = {k: v for k, v in nom_factores.items() if k != "Perfumista identificado"}
tabla_rep = coef_df(rep_in, nom_rep).join(coef_df(rep_out, nom_rep), lsuffix=" (dentro)", rsuffix=" (fuera)").round(3)
display(tabla_rep)

In [ ]:
a_in = coef_df(rep_in, {inv[c]: c for c in acc_safe})
a_out = coef_df(rep_out, {inv[c]: c for c in acc_safe})
sig_in = (a_in["lo"] > 0) | (a_in["hi"] < 0)
sig_out = (a_out["lo"] > 0) | (a_out["hi"] < 0)
mismo_signo = np.sign(a_in["coef"]) == np.sign(a_out["coef"])
print(f"Correlación de los efectos de los 44 acordes, dentro vs. fuera: {a_in['coef'].corr(a_out['coef']):.2f}")
print(f"Acordes significativos dentro: {sig_in.sum()} | de ellos con el mismo signo fuera: {(sig_in & mismo_signo).sum()} "
      f"| también significativos fuera: {(sig_in & sig_out).sum()}")

fig, ax = plt.subplots(figsize=(6.6, 6))
ax.scatter(a_in["coef"], a_out["coef"], c=np.where(sig_in & sig_out, "#15803d", "#94a3b8"), s=42)
lim = max(a_in["coef"].abs().max(), a_out["coef"].abs().max()) * 1.15
ax.plot([-lim, lim], [-lim, lim], color="#c2410c", ls="--", lw=1, label="Igual efecto")
ax.axhline(0, color="#cbd5e1", lw=0.8)
ax.axvline(0, color="#cbd5e1", lw=0.8)
for nombre in set(a_in["coef"].nlargest(5).index) | set(a_in["coef"].nsmallest(5).index):
    ax.annotate(nombre, (a_in.loc[nombre, "coef"], a_out.loc[nombre, "coef"]), fontsize=8, xytext=(4, 3), textcoords="offset points")
ax.set(xlabel="Efecto del acorde dentro de fra_cleaned", ylabel="Efecto del acorde fuera de fra_cleaned",
       title="Réplica: los acordes (verde = significativo en ambos)")
ax.legend(loc="upper left")
fig.tight_layout()
save(fig, "15_replica_acordes")

**Qué se sostiene y qué no**
- **Época:** se replica con claridad (los perfumes anteriores a 1990 puntúan más que los de la década de 2010), con magnitudes similares.
- **Acordes:** hay concordancia sustancial (correlación alta y casi todos los significativos con el mismo signo). Los más firmes,
  significativos en ambas muestras, incluyen vainilla, oud y ámbar (positivos) y marino y afrutado (negativos). Otros (tabaco, canela, iris)
  mantienen el signo pero pierden significancia: la muestra de afuera es 3 veces más chica.
- **Género:** "masculino > femenino" se replica con la mitad de magnitud; **"unisex > femenino" no se replica** (el intervalo incluye el cero).
  Esa parte del hallazgo depende de cómo se seleccionó el subconjunto y no la considero firme.
- Los ajustes son más bajos afuera (R² ajustado menor), coherente con una muestra más chica y sin país ni perfumista.

## 5. Parte 2 — Mercado: valoración vs. demanda

> **Nota sobre el precio.** Los dos archivos disponibles **no contienen precios**, por lo que no es posible responder
> "¿el precio acompaña a la valoración?". Esta parte analiza lo que sí se puede medir con estos datos: la **demanda**
> (usando la cantidad de votos como *proxy*) y su relación con la valoración. La sección 5.6 detalla qué haría falta
> para incorporar precio.
>
> **Cuidado con el proxy:** los votos miden atención de la comunidad de Fragrantica (entusiastas, sesgo hacia ciertos mercados),
> **no ventas**. Sirven para comparar popularidad relativa, no para estimar facturación.

### 5.1 La demanda está muy concentrada (catálogo completo, 70.000 perfumes)
Para esto se usa `fra_perfumes.csv`, que incluye también perfumes con muy pocos votos.

In [ ]:
votos = full["votes"].dropna().to_numpy()
v_asc = np.sort(votos)
v_desc = v_asc[::-1]

L = np.insert(np.cumsum(v_asc) / v_asc.sum(), 0, 0)
x_l = np.linspace(0, 1, len(L))
gini = 1 - 2 * np.trapezoid(L, x_l)

conc = {f"Top {int(q * 100)} % de perfumes": v_desc[: int(round(len(v_desc) * q))].sum() / v_desc.sum()
        for q in (0.01, 0.05, 0.10, 0.20)}

print(f"Perfumes en el catálogo: {len(full):,} | con al menos un voto: {len(votos):,} "
      f"| sin votos: {full['votes'].isna().mean():.1%}")
print(f"Votos por perfume — mediana: {np.median(votos):.0f} | media: {votos.mean():.0f} | máximo: {votos.max():.0f}")
print(f"Perfumes con menos de {min_votes} votos: {(votos < min_votes).mean():.1%}")
print(f"Índice de Gini de los votos: {gini:.3f}")
for k, v_ in conc.items():
    print(f"  {k} concentra el {v_:.1%} de todos los votos")

fig, ax = plt.subplots(figsize=(5.8, 5.2))
ax.plot(x_l, L, color="#1d4ed8", lw=2.2, label=f"Votos acumulados (Gini = {gini:.2f})")
ax.plot([0, 1], [0, 1], color="#94a3b8", ls="--", label="Reparto parejo")
ax.set(xlabel="Proporción acumulada de perfumes (de menos a más votados)",
       ylabel="Proporción acumulada de votos", title="Curva de Lorenz de la demanda")
ax.legend(loc="upper left")
fig.tight_layout()
save(fig, "09_lorenz_demanda")

La demanda sigue una ley de potencias: una porción pequeña de perfumes concentra casi toda la atención y la mayoría
(la mitad de los perfumes con votos) no llega al umbral de 26 votos. **Ser descubierto es el cuello de botella**, no solo ser bueno.

### 5.2 ¿La calidad percibida se traduce en demanda?
Se clasifica cada perfume según su valoración (score suavizado) y su demanda (votos), con los umbrales de la configuración
(`Q_QUALITY`, `Q_DEMAND`: percentil 75 por defecto). Se usan los 24.063 perfumes de `fra_cleaned`.

In [ ]:
q_q = cl["score"].quantile(Q_QUALITY)
q_d = cl["votes"].quantile(Q_DEMAND)
alta_q, alta_d = cl["score"] >= q_q, cl["votes"] >= q_d
cl["segmento"] = np.select([alta_q & alta_d, alta_q & ~alta_d, ~alta_q & alta_d],
                           ["Estrellas", "Joyas poco conocidas", "Masivos de valoración común"], "Resto")
cl["etiqueta"] = titulo(cl["brand"]) + " — " + titulo(cl["perfume"])

rho, p_rho = stats.spearmanr(cl["score"], cl["votes"])
print(f"Umbral de alta valoración: score ≥ {q_q:.2f} | umbral de alta demanda: ≥ {q_d:.0f} votos")
print(f"Correlación de Spearman valoración–votos: {rho:.3f} (p={p_rho:.2g})")

seg = cl.groupby("segmento").agg(perfumes=("score", "size"), votos_totales=("votes", "sum"),
                                 rating_medio=("rating", "mean"), votos_mediana=("votes", "median"))
seg["% de perfumes"] = seg["perfumes"] / seg["perfumes"].sum() * 100
seg["% de los votos"] = seg["votos_totales"] / seg["votos_totales"].sum() * 100
orden_seg = ["Estrellas", "Joyas poco conocidas", "Masivos de valoración común", "Resto"]
display(seg.loc[orden_seg, ["perfumes", "% de perfumes", "% de los votos", "rating_medio", "votos_mediana"]].round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.6))
hb = ax.hexbin(cl["log_votes"], cl["score"], gridsize=55, cmap="Blues", mincnt=1, bins="log")
ax.axvline(np.log10(q_d), color="#c2410c", ls="--", lw=1.3)
ax.axhline(q_q, color="#c2410c", ls="--", lw=1.3)
ax.text(np.log10(q_d) + 0.05, cl["score"].max() * 0.985, "Estrellas", color="#15803d", fontweight="bold", va="top")
ax.text(cl["log_votes"].min() + 0.03, cl["score"].max() * 0.985, "Joyas poco conocidas", color="#15803d", fontweight="bold", va="top")
ax.text(np.log10(q_d) + 0.05, q_q - 0.45, "Masivos de valoración común", color="#475569", fontweight="bold")
ax.set(xlabel="Votos (log10)", ylabel="Score suavizado",
       title="Valoración vs. demanda: casi no hay relación")
fig.colorbar(hb, ax=ax, label="Perfumes (escala log)")
fig.tight_layout()
save(fig, "10_valoracion_vs_demanda")

In [ ]:
cols_show = ["etiqueta", "rating", "votes", "score"]
ren = {"etiqueta": "Perfume", "rating": "Rating", "votes": "Votos", "score": "Score"}
print("ESTRELLAS — los 10 más votados:")
display(cl[cl["segmento"] == "Estrellas"].nlargest(10, "votes")[cols_show].rename(columns=ren).round(2).reset_index(drop=True))
print("JOYAS POCO CONOCIDAS — las 10 mejor valoradas:")
display(cl[cl["segmento"] == "Joyas poco conocidas"].nlargest(10, "score")[cols_show].rename(columns=ren).round(2).reset_index(drop=True))
q_low = cl["score"].quantile(1 - Q_QUALITY)
print(f"MASIVOS MAL VALORADOS — los 10 más votados con score ≤ {q_low:.2f} (cuartil inferior):")
display(cl[(cl["votes"] >= q_d) & (cl["score"] <= q_low)].nlargest(10, "votes")[cols_show].rename(columns=ren).round(2).reset_index(drop=True))

**Lectura.** La correlación entre valoración y votos es prácticamente nula: **ser muy bueno no garantiza ser muy visto**, y
ser muy visto tampoco implica ser muy bueno. Los "masivos de valoración común" (demanda alta, valoración no destacada) son el 17 %
de los perfumes y concentran más de la mitad de los votos; las "joyas poco conocidas" son otro 17 % y reciben alrededor del 4 %.
La última lista muestra perfumes muy populares con valoraciones en el cuartil inferior.

### 5.3 ¿Lo que mejora la valoración es lo mismo que atrae demanda?
Se repite el análisis de la Parte 1 usando como objetivo `log10(votos)`.

In [ ]:
res_dem = {
    "Demanda · marcas nuevas": cv_run(make_hgb, X, y_logv, gkf, brand),
    "Demanda · marca conocida": cv_run(make_hgb, X, y_logv, kf, use_brand=True),
}
display(pd.DataFrame(res_dem).T.round(3))

imp_dem = group_importance(X, y_logv, GRUPOS)
cmp_imp = pd.DataFrame({"Valoración": imp_score["caida_R2"], "Demanda": imp_dem["caida_R2"]})
cmp_norm = cmp_imp.clip(lower=0).div(cmp_imp.clip(lower=0).sum()) * 100
display(cmp_imp.round(4))

fig, ax = plt.subplots(figsize=(9, 4.2))
orden_g = cmp_norm.sort_values("Valoración", ascending=True).index
ypos = np.arange(len(orden_g))
ax.barh(ypos + 0.2, cmp_norm.loc[orden_g, "Valoración"], height=0.38, color="#1d4ed8", label="Valoración")
ax.barh(ypos - 0.2, cmp_norm.loc[orden_g, "Demanda"], height=0.38, color="#f59e0b", label="Demanda (votos)")
ax.set_yticks(ypos)
ax.set_yticklabels(orden_g)
ax.set(xlabel="% de la importancia total (marcas nuevas)", title="Qué empuja la valoración y qué empuja la demanda")
ax.legend()
fig.tight_layout()
save(fig, "11_importancia_valoracion_vs_demanda")

In [ ]:
# Dirección de los efectos sobre la demanda (descriptivo, votos medianos por segmento).
d_dem = d_ols.copy()
tab_perf = d_dem.groupby("perf_known")["votes"].agg(["median", "count"]).rename(index={0: "Perfumista sin identificar", 1: "Perfumista identificado"})
tab_dec = d_dem.groupby("decada_g")["votes"].agg(["median", "count"]).loc[["pre1980", "1980s", "1990s", "2000s", "2010s", "2020s", "sin_anio"]]
tab_gen = d_dem.groupby("gender")["votes"].agg(["median", "count"])
for nombre, t in [("Perfumista", tab_perf), ("Década", tab_dec), ("Género", tab_gen)]:
    print(f"Votos medianos por {nombre}:")
    display(t.rename(columns={"median": "votos_mediana", "count": "perfumes"}))

**Lectura.** Los determinantes **no coinciden**: la valoración depende sobre todo del perfil olfativo (acordes), el género y el país,
mientras que la demanda depende más de la **época** (los lanzamientos más antiguos acumularon más votos con los años) y de si el
**perfumista está identificado** (posible señal de marcas con más trayectoria). Esto sugiere (sin probarlo)
que *qué se huele* mueve la satisfacción y que la trayectoria y el posicionamiento mueven la visibilidad.
Son asociaciones observacionales: no prueban que cambiar una variable cause el resultado.

### 5.4 Marcas: tamaño de portafolio, valoración y atención

In [ ]:
br = cl.groupby("brand").agg(perfumes=("score", "size"), score_medio=("score", "mean"),
                             votos_total=("votes", "sum"), votos_mediana=("votes", "median"))
br["marca"] = titulo(br.index.to_series())
br5 = br[br["perfumes"] >= 5]
rho_b, p_b = stats.spearmanr(np.log10(br5["perfumes"]), br5["score_medio"])
top10_votos = br["votos_total"].nlargest(10).sum() / br["votos_total"].sum()
print(f"Marcas: {len(br):,} | con ≥5 perfumes: {len(br5)} | con ≥{MIN_BRAND_PERFUMES}: {(br['perfumes'] >= MIN_BRAND_PERFUMES).sum()}")
print(f"Correlación (Spearman) tamaño de portafolio vs. valoración media, marcas con ≥5 perfumes: {rho_b:.3f} (p={p_b:.2g})")
print(f"Las 10 marcas con más votos concentran el {top10_votos:.1%} de los votos del dataset")

big = br[br["perfumes"] >= MIN_BRAND_PERFUMES].sort_values("score_medio")
sel = pd.concat([big.head(10), big.tail(10)])
fig, ax = plt.subplots(figsize=(8.5, 6))
ax.barh(sel["marca"], sel["score_medio"], color=["#c2410c"] * 10 + ["#15803d"] * 10)
ax.axvline(cl["score"].mean(), color="#64748b", ls="--", lw=1, label="Promedio global")
ax.set_xlim(3.4, 4.5)
ax.set(xlabel="Score medio de la marca", title=f"Marcas peor y mejor valoradas (≥{MIN_BRAND_PERFUMES} perfumes)")
ax.legend(loc="lower right")
fig.tight_layout()
save(fig, "12_marcas_ranking")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))
sc = ax.scatter(br5["perfumes"], br5["score_medio"], s=np.sqrt(br5["votos_total"]) / 4, alpha=0.45, color="#1d4ed8", edgecolor="white", lw=0.4)
ax.set_xscale("log")
ax.axhline(cl["score"].mean(), color="#64748b", ls="--", lw=1)
for nombre, r in br5.nlargest(10, "votos_total").iterrows():
    ax.annotate(r["marca"], (r["perfumes"], r["score_medio"]), fontsize=7.5, xytext=(4, 3), textcoords="offset points")
ax.set(xlabel="Perfumes en el catálogo de la marca (escala log)", ylabel="Score medio de la marca",
       title="Portafolio grande ≠ mejor valoración (tamaño del punto = votos totales)")
fig.tight_layout()
save(fig, "13_marcas_portafolio")

### 5.5 Oferta vs. valoración vs. demanda por segmento
Para cada acorde y para cada género: qué proporción del catálogo lo ofrece (**oferta**), cómo se valora y cuánta atención recibe.
Es una lectura descriptiva para **generar hipótesis**, no una recomendación de lanzamiento.

In [ ]:
filas = []
for col in GRUPOS["Acordes"]:
    mask = X[col] == 1
    filas.append({"acorde": col.replace("acc_", ""), "oferta_%": mask.mean() * 100,
                  "score_medio": df.loc[mask, "score"].mean(), "votos_mediana": df.loc[mask, "votes"].median()})
ac = pd.DataFrame(filas).set_index("acorde")

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(ac["oferta_%"], ac["score_medio"], s=ac["votos_mediana"] * 2.2, alpha=0.5, color="#1d4ed8", edgecolor="white")
ax.axhline(df["score"].mean(), color="#64748b", ls="--", lw=1)
ax.axvline(ac["oferta_%"].median(), color="#64748b", ls="--", lw=1)
etiquetar = set(ac["score_medio"].nlargest(6).index) | set(ac["score_medio"].nsmallest(6).index) | set(ac["oferta_%"].nlargest(5).index)
for nombre in etiquetar:
    ax.annotate(nombre, (ac.loc[nombre, "oferta_%"], ac.loc[nombre, "score_medio"]), fontsize=8, xytext=(5, 4), textcoords="offset points")
ax.set(xlabel="Oferta: % de perfumes que tienen el acorde", ylabel="Score medio de esos perfumes",
       title="Acordes: oferta vs. valoración (tamaño = votos medianos)")
fig.tight_layout()
save(fig, "14_acordes_oferta_valoracion")

top_hip = ac[(ac["oferta_%"] < ac["oferta_%"].median()) & (ac["score_medio"] > df["score"].mean())].sort_values("score_medio", ascending=False)
print("Acordes con oferta por debajo de la mediana y valoración sobre el promedio (hipótesis a validar):")
display(top_hip.round(2))

gen = df.groupby("gender").agg(perfumes=("score", "size"), score_medio=("score", "mean"), votos_mediana=("votes", "median"), votos_totales=("votes", "sum"))
gen["oferta_%"] = gen["perfumes"] / gen["perfumes"].sum() * 100
gen["% de los votos"] = gen["votos_totales"] / gen["votos_totales"].sum() * 100
display(gen[["perfumes", "oferta_%", "score_medio", "votos_mediana", "% de los votos"]].round(2))

### 5.6 Qué falta para analizar el precio
Con precio se podría responder la pregunta original (*¿el precio acompaña a la valoración?*): precio vs. score, marcas
sobre/subvaloradas respecto de su precio, y segmentos "baratos y bien valorados".

**Qué se necesita:** una tabla con **marca, nombre del perfume y precio** (idealmente con tamaño/concentración, moneda y fecha).
Una opción pública es el *Perfume E-Commerce Dataset 2024* de Kaggle (listados de eBay), pero tiene solo ~2.000 listados y
requiere un cruce por nombre y marca (*fuzzy matching*) con riesgo de errores; una fuente de precios de catálogo sería preferible.
El cruce con este análisis se haría por la **marca normalizada** (en `fra_cleaned` las marcas ya vienen como *slug*,
por ejemplo `jean-paul-gaultier`) y luego por nombre de perfume.

## 6. Conclusiones

### Hallazgos
1. **La valoración de un perfume se explica poco por sus características.** Con marcas que el modelo nunca vio, acordes, notas,
   género, país, época y perfumista explican alrededor del 15 % de la varianza (R² ≈ 0,15). Conocer la **marca** lo duplica (≈ 0,30):
   la marca aporta tanto como todo el resto junto.
2. **Qué se asocia a mejor valoración** (diferencias pequeñas, de 0,03 a 0,15 puntos sobre 5):
   - **Acordes:** vainilla, oud y ámbar puntúan más; marino y afrutado, menos. Estos efectos se sostienen dentro de la misma marca
     y en perfumes fuera del subconjunto original.
   - **Época:** los perfumes anteriores a 1990 puntúan más; probablemente por sesgo de supervivencia, no por mejor calidad.
   - **Género:** "masculino > femenino" se sostiene con magnitud menor en la réplica; "unisex > femenino" **no** es robusto.
   - **País de la marca:** Brasil y países del Golfo puntúan más que Francia; Polonia, Rusia y Suecia, menos (no se pudo replicar: el
     archivo grande no trae país).
3. **La demanda está muy concentrada y casi no depende de la calidad percibida.** El 10 % de los perfumes más votados concentra
   más del 75 % de los votos; la correlación valoración–votos es ≈ 0,05. Hay muchas "joyas poco conocidas" y muchos perfumes populares
   con valoración común.
4. **Lo que mueve la valoración no es lo que mueve la demanda.** La valoración depende sobre todo del perfil olfativo, género y país;
   la demanda, de la época de lanzamiento y de si el perfumista está identificado.

### Hipótesis para una marca (a validar, no recomendaciones)
- El perfil olfativo ayuda a ser mejor valorado, pero **no** a ser visto: la visibilidad parece un problema distinto.
- Hay acordes bien valorados con poca oferta (tabaco, canela, balsámico, ahumado, iris), que serían candidatos a explorar
  **si** hubiera evidencia de demanda y precio.
- El segmento femenino es el de mayor oferta (≈ 47 %) y el de menor valoración media: un mercado saturado y de valoración exigente.

### Limitaciones
- **Votos ≠ ventas.** La comunidad de Fragrantica es de entusiastas y no representa al comprador promedio.
- **`fra_cleaned` no es una muestra aleatoria** y todo aplica a perfumes con ≥ 26 votos (ver 1.2 y 4.5).
- **Datos observacionales:** son asociaciones, no efectos causales; la época está confundida con la supervivencia de los clásicos.
- **Poder explicativo bajo** (R² ≈ 0,15 sin marca): útil para entender patrones, no para predecir con precisión un perfume individual.
- **`SHRINK_M` es una decisión del analista**; la sensibilidad (4.4) muestra que no cambia las conclusiones.
- **No hay precio**: no se responde si el precio acompaña a la valoración.

### Próximos pasos
1. Incorporar una fuente de **precios** y cruzar por marca y nombre (sección 5.6).
2. Probar un modelo ordinal o con efectos aleatorios por marca, más fiel a la estructura de los datos.
3. Medir el sesgo de la comunidad con otra fuente de valoraciones (por ejemplo, reseñas de e-commerce).